In [1]:
# merge_shards_to_single_npz.py
import numpy as np
import glob

def merge_shards(out_path, shard_glob, shuffle=True, seed=42, compress=True, astype=None):
    paths = sorted(glob.glob(shard_glob))
    assert paths, f"No shards match: {shard_glob}"
    Xs, Ys = [], []
    axes_ref = None
    for p in paths:
        d = np.load(p, mmap_mode='r')
        X, Y, axes = d["X"], d["Y"], str(d["axes"])
        if axes_ref is None:
            axes_ref = axes
            ref_shape = X.shape[1:]
        else:
            assert axes == axes_ref, f"axes mismatch: {p} has {axes}, expected {axes_ref}"
            assert X.shape[1:] == ref_shape, f"shape mismatch in {p}: {X.shape} vs {ref_shape}"
        if astype is not None:
            X = X.astype(astype, copy=False)
            Y = Y.astype(astype, copy=False)
        Xs.append(X)
        Ys.append(Y)

    X_all = np.concatenate(Xs, axis=0)
    Y_all = np.concatenate(Ys, axis=0)

    if shuffle:
        rng = np.random.default_rng(seed)
        perm = rng.permutation(X_all.shape[0])
        X_all = X_all[perm]
        Y_all = Y_all[perm]

    saver = np.savez_compressed if compress else np.savez
    saver(out_path, X=X_all, Y=Y_all, axes=axes_ref)
    print(f"Saved {out_path}\n  X:{X_all.shape}  Y:{Y_all.shape}  axes:{axes_ref}  dtype:{X_all.dtype}")

# Example usage:

merge_shards("./data/training_merged.npz", "./data/shards/patches_*.npz",
              shuffle=True, seed=42, compress=True, astype='float32')  # or 'float16' to reduce RAM


Saved ./data/training_merged.npz
  X:(19656, 1, 96, 32, 32)  Y:(19656, 1, 96, 32, 32)  axes:SCZYX  dtype:float32


In [14]:

# CARE config (same as yours)
config = Config(
    axes=axes,
    n_channel_in=n_in,
    n_channel_out=n_out,
    probabilistic=False,
    unet_residual=True,
    unet_n_depth=3,
    unet_kern_size=3,
    unet_n_first=32,
    train_epochs=30,
    train_steps_per_epoch=100,   # with generators, this is used by fit()
)

model = CARE(config, name='my_model', basedir='models')

# Build generators
train_gen = NPZMultiShardSequence(train_shards, batch_size=4, shuffle=True)
val_gen   = NPZMultiShardSequence(val_shards,   batch_size=4, shuffle=False)

history = model.keras_model.fit(
    data_train,
    validation_data=data_val,
    epochs=config.train_epochs,
    steps_per_epoch=config.train_steps_per_epoch,
    validation_steps=min(len(data_val), 50) if data_val else None
)


ValueError: You must call `compile()` before using the model.